<img src="../static/imo_health.png" alt="IMO Health Logo" width="300"/>

--
## IMO Coding Intelligence API — Notebook 1: Setup & Authentication

This notebook walks through the first step of the Coding Intelligence pipeline: **installing dependencies and authenticating with the IMO Health API**.

The IMO Coding Intelligence API uses **OAuth 2.0 client credentials** flow. Before any API call can be made, you must exchange your `client_id` and `client_secret` for a short-lived bearer token.

---

### What you'll cover
1. Install required Python packages
2. Configure your IMO API credentials
3. Obtain a bearer token from the IMO OAuth endpoint
4. Verify the token is working by inspecting the response

>  **Prerequisite:** You need valid IMO Coding Intelligence API credentials (`client_id` and `client_secret`). Contact IMO Health to obtain these.


## Step 1: Install Required Packages

In [ ]:
%pip install requests pandas --quiet
print("✓ Packages installed")

In [ ]:
import requests
import json
import pandas as pd
from datetime import datetime

print("✓ Imports ready")

## Step 2: Configure API Credentials

Load credentials from `config.json` (same folder as this notebook).

Expected structure:
- `auth0.client_id`
- `auth0.client_secret`
- `auth0.audience` (optional; defaults to `https://api.imohealth.com`)

> ⚠️ **Never commit credentials to source control.** In production, load these from environment variables or a secrets manager.

In [ ]:
from pathlib import Path
import json

# Locate config.json in common working-directory scenarios
candidate_paths = [
    Path("config.json"),
    Path("notebooks/config.json"),
    Path("CodingIntelligence/notebooks/config.json"),
]

config_path = next((p for p in candidate_paths if p.exists()), None)
if config_path is None:
    raise FileNotFoundError(
        "config.json not found. Expected at notebooks/config.json or current working directory."
    )

with config_path.open("r", encoding="utf-8") as f:
    cfg = json.load(f)

auth0_cfg = cfg.get("auth0", {})
CLIENT_ID = auth0_cfg.get("client_id", "").strip()
CLIENT_SECRET = auth0_cfg.get("client_secret", "").strip()
AUDIENCE = auth0_cfg.get("audience", "https://api.imohealth.com")

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError(
        "Missing auth0.client_id or auth0.client_secret in config.json"
    )

# IMO OAuth & API endpoints
AUTH_URL = "https://api.imohealth.com/oauth/token"
CODING_INTEL_URL = "https://api.imohealth.com/codingintelligence/v1/rules/imo-admin-coding-sets"
EXCLUDES1_URL = "https://api.imohealth.com/codingintelligence/v1/rules/cms-excludes1"

print(f"Loaded config: {config_path}")
print(f"Client ID configured: {'✓' if CLIENT_ID else '✗'}")
print(f"Client Secret configured: {'✓' if CLIENT_SECRET else '✗'}")
print(f"Audience: {AUDIENCE}")

## Step 3: Obtain a Bearer Token

The IMO API uses the **OAuth 2.0 client credentials** grant. We POST our `client_id` and `client_secret` to the token endpoint and receive a short-lived `access_token` (~18 hours).

This token is attached as a `Bearer` header on every subsequent API call.

In [ ]:
def get_access_token(client_id: str, client_secret: str) -> str:
    """
    Exchange client credentials for an IMO OAuth bearer token.
    
    Returns:
        str: The access_token to use in Authorization headers.
    """
    payload = {
        "grant_type":    "client_credentials",
        "client_id":     client_id,
        "client_secret": client_secret,
        "audience":      "https://api.imohealth.com"
    }

    response = requests.post(AUTH_URL, json=payload, timeout=30)
    response.raise_for_status()

    data       = response.json()
    token      = data["access_token"]
    expires_in = data.get("expires_in", 3600)

    print(f"✓ Access token obtained")
    print(f"  Token type : {data.get('token_type', 'Bearer')}")
    print(f"  Expires in : {expires_in}s (~{expires_in // 3600}h {(expires_in % 3600) // 60}m)")
    print(f"  Token preview : {token[:30]}...")
    return token


# Obtain the token — store it for use in later cells
ACCESS_TOKEN = get_access_token(CLIENT_ID, CLIENT_SECRET)

## Step 4: Build Reusable Headers

Once you have a token, you'll use the same `Authorization: Bearer <token>` header on every API call. We build that dict once here so every notebook cell can reuse it.

In [ ]:
# Build the headers dict — reuse this in Notebooks 02 and 03
HEADERS = {
    "Content-Type":  "application/json",
    "Authorization": f"Bearer {ACCESS_TOKEN}"
}

print("✓ Headers ready for API calls")
print(f"  Authorization: Bearer {ACCESS_TOKEN[:30]}...")

## ✅ Summary

| Step | Description | Status |
|------|-------------|--------|
| 1 | Install packages (`requests`, `pandas`) | ✓ |
| 2 | Configure `CLIENT_ID` and `CLIENT_SECRET` | ✓ |
| 3 | POST to OAuth endpoint → receive `access_token` | ✓ |
| 4 | Build `HEADERS` dict for subsequent API calls | ✓ |

---

**Next:** Open `02_Upload_Codes_and_Validate.ipynb` to load medical codes and fire the Coding Intelligence validation API.